<a href="https://colab.research.google.com/github/vcardui/RealTimeCharacterDetection/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [80]:
# +----------------------------------------------------------------------------+
# | CARDUI TECH v1.0.0
# +----------------------------------------------------------------------------+
# | Copyright (c) 2026 - 2026, CARDUITECH.COM (www.carduitech.com)
# | Vanessa Reteguín <vanessa@reteguin.com>
# | Released under the MIT license
# | www.carduitech.com/license/
# +----------------------------------------------------------------------------+
# | Author.......: Vanessa Reteguín <vanessa@reteguin.com>
# | First release: September 16th, 2026
# | Last update..: September 16th, 2026
# | Course.......: Automatons II
# | WhatIs.......: Real Time Character Detection - Main
# +----------------------------------------------------------------------------++
# ------------------------- Instructions -----------------------
# Instrucciones: crea una CNN para clasificar caracteres (conjunto de datos
# "mnist" o "emnist"). Utiliza la red para detectar caracteres en tiempo real.

# ---

# - Detection de caracteres en tiempo real
# - Palabra por palabra vs carácter a carácter

#    Hola = hola = hola.

# - Justificar uso de modelo: Transformer? CNN? Otra?
# - Conclusiones
# - *Entrenar por partes -> Check points

# ------------ Resources / Documentation involved -------------
# MNIST Dataset: https://www.kaggle.com/datasets/hojjatk/mnist-dataset

# Deep learning on MNIST: https://numpy.org/numpy-tutorials/tutorial-deep-learning-on-mnist/

In [81]:
# ------------------------- Libraries -------------------------
from kaggle.api.kaggle_api_extended import KaggleApi
import os  # os.path.exists(path)

import numpy as np
import matplotlib.pyplot as plt

In [82]:
# ------------------------- Variables -------------------------
DATASET_FOLDER = 'input/mnist-dataset/'
DATASET_ID = 'hojjatk/mnist-dataset'

data_sources = {
    "training_images": "train-images-idx3-ubyte",  # 60,000 training images.
    "test_images": "t10k-images-idx3-ubyte",  # 10,000 test images.
    "training_labels": "train-labels-idx1-ubyte",  # 60,000 training labels.
    "test_labels": "t10k-labels-idx1-ubyte",  # 10,000 test labels.
}

# Kaggle API

In [83]:
!pip install kaggle -q

from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

!rm -rf ~/.kaggle
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!ls ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json
User uploaded file "kaggle.json" with length 62 bytes
/root/.kaggle/kaggle.json


In [84]:
import kagglehub

# Download latest version
DATASET_FOLDER = kagglehub.dataset_download("hojjatk/mnist-dataset")
print(DATASET_FOLDER)

Using Colab cache for faster access to the 'mnist-dataset' dataset.
/kaggle/input/mnist-dataset


Save images and labels into mnist_dataset dictionary as 1D arrays (neural network will expect a 1D array)

Since each image is 28 x 28 pixels, each file is reshaped into a 1 x 784 array

rb = raw binary

In [85]:
mnist_dataset = {}

for key in ("training_images", "test_images"):
    with open(os.path.join(DATASET_FOLDER, f'{data_sources[key]}/{data_sources[key]}'), "rb") as mnist_file:
        mnist_dataset[key] = np.frombuffer(
            mnist_file.read(), np.uint8, offset=16
        ).reshape(-1, 28 * 28)

for key in ("training_labels", "test_labels"):
    with open(os.path.join(DATASET_FOLDER, f'{data_sources[key]}/{data_sources[key]}'), "rb") as mnist_file:
        mnist_dataset[key] = np.frombuffer(mnist_file.read(), np.uint8, offset=8)

Split dataset into training ant set test

In [86]:
x_train, y_train, x_test, y_test = (
    mnist_dataset["training_images"],
    mnist_dataset["training_labels"],
    mnist_dataset["test_images"],
    mnist_dataset["test_labels"],
)

print(f"""Dataset shapes:
training_images (x_train): {x_train.shape}
training labels (y_train): {y_train.shape}

test images (x_test): {x_test.shape}
test labels (y_test): {y_test.shape}
""")

Dataset shapes:
training_images (x_train): (60000, 784)
training labels (y_train): (60000,)

test images (x_test): (10000, 784)
test labels (y_test): (10000,)



Output: (60000, 784) -> (60000 rows, 3 784) -> 2D